### Import packages


In [6]:
pip install geopy scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
#!pip install geopy

import pandas as pd
import geopy.distance
import math
from scipy.spatial import cKDTree

### Define dataframes

In [8]:
# Training data
ground_df = pd.read_csv("Training_data_uhi_index.csv")

# Building footprint data (centroids)
centroid_df = pd.read_csv("Building_Footprint_centroid.csv")

# Test data
test_df = pd.read_csv("submission.csv")


### Define functions

In [9]:
# Calculate distances between two lat long coordinates
def haversine(lat1, lon1, lat2, lon2):
    # Radius of the Earth in kilometers
    R = 6371.0
    
    # Convert latitude and longitude from degrees to radians
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)
    
    # Difference in coordinates
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    # Haversine formula
    a = math.sin(dlat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    
    # Distance in kilometers
    distance = R * c
    return distance

# Example usage
#lat1, lon1 = 52.2296756, 21.0122287  # Coordinates of Warsaw, Poland
#lat2, lon2 = 41.8919300, 12.5113300  # Coordinates of Rome, Italy
#distance should be ~1315 km
#print(f"Distance: {haversine(lat1, lon1, lat2, lon2)} km")

# Function to calculate distances using a KD-Tree for efficient querying
def calculate_distances(ground_df, centroid_df, num_nearest=100):
    # Step 1: Create a cKDTree for centroid_df (centroid coordinates)
    centroid_coords = centroid_df[['Y', 'X']].values  # Assuming 'Y' is latitude and 'X' is longitude
    centroid_tree = cKDTree(centroid_coords)
    
    # Step 2: Prepare a list to store the distances
    distances = []

    # Step 3: Loop through each row in ground_df and query the nearest centroids
    for _, ground_row in ground_df.iterrows():
        lat1 = ground_row['Latitude']
        lon1 = ground_row['Longitude']
        
        # Query the KD-Tree for the distances to the nearest 'num_nearest' centroids
        dist, indices = centroid_tree.query([lat1, lon1], k=num_nearest)  # Get the top 'num_nearest' distances
        
        # Step 4: Loop over the results and calculate the exact distance for each centroid
        for i, idx in enumerate(indices):
            lat2 = centroid_df.iloc[idx]['Y']
            lon2 = centroid_df.iloc[idx]['X']
            haversine_dist = haversine(lat1, lon1, lat2, lon2)

            distances.append({
                'ground_id': ground_row.name,  # Row index from ground_df
                'centroid_id': centroid_df.iloc[idx]['fid'],  # Using 'fid' for the centroid ID
                'distance_km': haversine_dist
            })

    # Step 5: Create a DataFrame to hold the distances
    distance_df = pd.DataFrame(distances)
    return distance_df


# Function to calculate the number of buildings within a specified distance (in km) and return as a new DataFrame
def calculate_buildings_within_distance(ground_df, distance_df, max_distance_km=1.0):
    # Step 1: Filter the distance_df for distances <= max_distance_km
    within_distance = distance_df[distance_df['distance_km'] <= max_distance_km]

    # Step 2: Group by 'ground_id' (which is now the index) and count the number of buildings (centroid_ids) within the max distance
    buildings_within_distance = within_distance.groupby('ground_id').size().reset_index(name='buildings_within_distance')

    # Step 3: Merge the counts into a new DataFrame (using the index as ground_id)
    result_df = ground_df.merge(buildings_within_distance, left_index=True, right_on='ground_id', how='left')

    # Step 4: If any ground_id has no buildings within the specified distance, fill it with 0
    result_df['buildings_within_distance'] = result_df['buildings_within_distance'].fillna(0).astype(int)

    # Drop the ground_id column as it's no longer needed after merging
    result_df = result_df.drop(columns='ground_id')

    return result_df

### Transforming training data

In [10]:
# Example usage (assuming you already have distance_df and ground_df)
# Calculate the number of buildings within k-disrtance for each ground point and return the result as a new DataFrame
ground_distance_df = calculate_distances(ground_df, centroid_df, num_nearest=100)

ground_nearest_building_count_10m = calculate_buildings_within_distance(ground_df, ground_distance_df, max_distance_km=0.01)
ground_nearest_building_count_50m = calculate_buildings_within_distance(ground_df, ground_distance_df, max_distance_km=0.05)
ground_nearest_building_count_100m = calculate_buildings_within_distance(ground_df, ground_distance_df, max_distance_km=0.1)
ground_nearest_building_count_500m = calculate_buildings_within_distance(ground_df, ground_distance_df, max_distance_km=0.5)

ground_nearest_building_count_10m = ground_nearest_building_count_10m.reset_index()
ground_nearest_building_count_50m = ground_nearest_building_count_50m.reset_index()
ground_nearest_building_count_100m = ground_nearest_building_count_100m.reset_index()
ground_nearest_building_count_500m = ground_nearest_building_count_500m.reset_index()

ground_df["nearest_building_count_10m"] = ground_nearest_building_count_10m["buildings_within_distance"]
ground_df["nearest_building_count_50m"] = ground_nearest_building_count_50m["buildings_within_distance"]
ground_df["nearest_building_count_100m"] = ground_nearest_building_count_100m["buildings_within_distance"]
ground_df["nearest_building_count_500m"] = ground_nearest_building_count_500m["buildings_within_distance"]

ground_df.head()

,Longitude,Latitude,datetime,UHI Index,nearest_building_count_10m,nearest_building_count_50m,nearest_building_count_100m,nearest_building_count_500m
0,-73.909167,40.813107,24-07-2021 15:53,1.030289,0,0,5,100
1,-73.909187,40.813045,24-07-2021 15:53,1.030289,0,0,5,100
2,-73.909215,40.812978,24-07-2021 15:53,1.023798,0,0,4,100
3,-73.909242,40.812908,24-07-2021 15:53,1.023798,0,0,4,100
4,-73.909257,40.812845,24-07-2021 15:53,1.021634,0,0,4,100


### Transforming validatation data

In [11]:
test_distance_df = calculate_distances(test_df, centroid_df, num_nearest=100)

test_df_nearest_building_count_10m = calculate_buildings_within_distance(test_df, test_distance_df, max_distance_km=0.01)
test_df_nearest_building_count_50m = calculate_buildings_within_distance(test_df, test_distance_df, max_distance_km=0.05)
test_df_nearest_building_count_100m = calculate_buildings_within_distance(test_df, test_distance_df, max_distance_km=0.1)
test_df_nearest_building_count_500m = calculate_buildings_within_distance(test_df, test_distance_df, max_distance_km=0.5)

test_df_nearest_building_count_10m = test_df_nearest_building_count_10m.reset_index()
test_df_nearest_building_count_50m = test_df_nearest_building_count_50m.reset_index()
test_df_nearest_building_count_100m = test_df_nearest_building_count_100m.reset_index()
test_df_nearest_building_count_500m = test_df_nearest_building_count_500m.reset_index()

test_df["nearest_building_count_10m"] = test_df_nearest_building_count_10m["buildings_within_distance"]
test_df["nearest_building_count_50m"] = test_df_nearest_building_count_50m["buildings_within_distance"]
test_df["nearest_building_count_100m"] = test_df_nearest_building_count_100m["buildings_within_distance"]
test_df["nearest_building_count_500m"] = test_df_nearest_building_count_500m["buildings_within_distance"]



In [12]:
# Save the transformed dataframes to CSV files
ground_df.to_csv('Training_data_uhi_index.csv', index=False)
test_df.to_csv('submission.csv', index=False)
